In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch

/Users/eivinas/dev/ccn-abstract-clustering/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
N_CLUSTERS = 10
N_UMAP_COMPONENTS = 5

In [4]:
df = pd.read_csv("ccn-2026-pending-posters.csv")
abstracts = df['abstract'].values
len(abstracts)

617

In [5]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [6]:
from pathlib import Path

if Path("ccn_embeddings.npy").exists():
    embedding_model = None
    print("Embeddings cache found — skipping model load.")
else:
    embedding_model = SentenceTransformer(
        "BAAI/bge-multilingual-gemma2",
        model_kwargs={"torch_dtype": torch.float16}
    ).to(device)
    print("Embedding model loaded.")

Embeddings cache found — skipping model load.


In [7]:
import numpy as np
from pathlib import Path

CLUSTERING_PROMPT = "Identify the primary cognitive neuroscience, deep learning, or neuroscience domain of this study." \
                    "Focusing particularly on the subtrack of the cognitive computational neuroscience subtrack:"
EMBEDDINGS_PATH = Path("ccn_embeddings.npy")

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Loaded embeddings from {EMBEDDINGS_PATH} — shape: {embeddings.shape}")
else:
    print(f"Encoding {len(abstracts)} abstracts with Gemma embeddings...")
    embeddings = embedding_model.encode(
        list(abstracts),
        prompt=CLUSTERING_PROMPT,
        show_progress_bar=True,
        batch_size=16,
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Saved embeddings to {EMBEDDINGS_PATH} — shape: {embeddings.shape}")

Loaded embeddings from ccn_embeddings.npy — shape: (617, 3584)


In [8]:
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import BaseRepresentation
from dotenv import load_dotenv
import anthropic
import time
import os

load_dotenv()

class ClaudeRepresentation(BaseRepresentation):
    """BERTopic representation model that uses Claude to name each topic."""

    def __init__(self, client, model="claude-haiku-4-5", prompt_template=None, nr_docs=20, delay_in_seconds=0.5):
        self.client = client
        self.model = model
        self.prompt_template = prompt_template
        self.nr_docs = nr_docs
        self.delay_in_seconds = delay_in_seconds

    def extract_topics(self, topic_model, documents, c_tf_idf, topics):
        repr_docs = topic_model.representative_docs_
        updated_topics = {}
        for topic_id in topics:
            docs = repr_docs.get(topic_id, [])[:self.nr_docs]
            prompt = self.prompt_template.replace("[DOCUMENTS]", "\n\n---\n\n".join(docs))
            response = self.client.messages.create(
                model=self.model,
                max_tokens=30,
                messages=[{"role": "user", "content": prompt}],
            )
            label = response.content[0].text.strip()
            updated_topics[topic_id] = [(label, 1)]
            time.sleep(self.delay_in_seconds)
        return updated_topics


# {N_DIMS_UMAP}D UMAP for clustering
umap_model = UMAP(n_components=N_UMAP_COMPONENTS, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)

# K-Means with exactly 7 — balance comes for free, no nr_topics merging step needed
cluster_model = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")

vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3)

claude_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# ccn_prompt = """You are an Area Chair at the Cognitive Computational Neuroscience (CCN) conference.
# Below are a few representative abstracts from a cluster of submitted posters.
# Reply with ONLY a concise 2-4 word conference track title — nothing else.

# Something that would be good for quickly skimming and understanding where you would place your own poster if it were in this cluster.

# Abstracts:
# [DOCUMENTS]

# Track title:"""

# representation_model = ClaudeRepresentation(
#     client=claude_client,
#     model="claude-opus-4-6",
#     prompt_template=ccn_prompt,
#     nr_docs=30,
# )

# No nr_topics — K-Means already gives exactly 7, nothing to merge
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    verbose=True,
)

In [9]:
# Step 1: cluster — fast, no API calls
topics, probs = topic_model.fit_transform(list(abstracts), embeddings)
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]])

2026-06-11 14:33:30,457 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-11 14:33:34,501 - BERTopic - Dimensionality - Completed ✓
2026-06-11 14:33:34,502 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-11 14:33:34,539 - BERTopic - Cluster - Completed ✓
2026-06-11 14:33:34,543 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-11 14:33:34,612 - BERTopic - Representation - Completed ✓


   Topic  Count                                      Name
0      0    128              0_neural_brain_task_activity
1      1     82       1_learning_participants_memory_task
2      2     71                2_brain_visual_neural_data
3      3     71  3_neural_networks_model_representational
4      4     70            4_learning_agents_model_models
5      5     43              5_models_language_llms_human
6      6     41         6_speech_language_auditory_neural
7      7     40           7_neural_systems_learning_model
8      8     36              8_visual_models_vision_human
9      9     35               9_visual_models_human_model


In [10]:
len(list(abstracts))

617

In [11]:
import numpy as np

def representative_docs(embeddings, topics, abstracts, n=12):
    topics = np.asarray(topics)
    reps = {}
    for t in sorted(set(topics)):
        idx = np.where(topics == t)[0]
        sub = embeddings[idx] / np.linalg.norm(embeddings[idx], axis=1, keepdims=True)
        c   = sub.mean(0); c /= np.linalg.norm(c)
        reps[t] = [abstracts[i] for i in idx[np.argsort(-(sub @ c))[:n]]]
    return reps

In [12]:
import json

reps = representative_docs(embeddings, topics, abstracts, n=25)
topics_arr = np.asarray(topics)

blocks = [
    f"CLUSTER {t} ({(topics_arr==t).sum()} posters):\n"
    + "\n".join(f"  - {d[:300]}" for d in docs)
    for t, docs in reps.items()
]

naming_prompt = f"""You are organizing the CCN 2026 reception. Researchers walk to one of
{len(reps)} bars based on their topic. Below are representative abstracts from each cluster.

{chr(10).join(blocks)}

Give each cluster a SHORT name (2-3 words) a researcher instantly recognizes as their own
area — like "Vision", "Decision-making", "Neural circuits", "Encoding/decoding".

You can see ALL clusters together: make the names MUTUALLY EXCLUSIVE. Where two clusters
look similar, find the real distinction and name it. No generic filler ("Neural Mechanisms",
"Representations", "Models") that could apply to any cluster.

Please use recognizable names like ("computational cognitive science", "vision", "decision-making", "neural circuits", "encoding/decoding", "representational geometry", etc. etc.)

Also list any pair of clusters that are weakly separated — bars where people won't know
which is theirs.

Return ONLY JSON: {{"labels": {{"0": "...", ...}}, "confusable_pairs": [[0,3], ...]}}"""

resp = claude_client.messages.create(
    model="claude-opus-4-8", max_tokens=10000,
    messages=[{"role": "user", "content": naming_prompt + "\n\nUse plain ASCII only. No quotation marks, colons, or special characters inside any name."}],
)
raw = resp.content[0].text
out = json.loads(raw[raw.find("{") : raw.rfind("}") + 1])

# out = json.loads(resp.content[0].text)
topic_model.set_topic_labels({int(k): v for k, v in out["labels"].items()})
print(out["confusable_pairs"])

[[0, 3], [0, 1], [1, 4], [2, 6], [5, 7], [8, 9], [2, 9]]


In [13]:
print(topic_model.get_topic_info()[["Topic", "Count", "CustomName"]])

   Topic  Count                    CustomName
0      0    128  Cognition and Memory Systems
1      1     82    Decision and Metacognition
2      2     71   Naturalistic Brain Encoding
3      3     71    Neural Population Dynamics
4      4     70        Reinforcement Learning
5      5     43            LLMs and Reasoning
6      6     41         Language Neuroscience
7      7     40         Neural Network Theory
8      8     36        Computer Vision Models
9      9     35          Visual Cortex Models


In [14]:
import matplotlib.pyplot as plt
import plotly.express as px
import textwrap
from umap import UMAP as UMAP2D

umap_2d = UMAP2D(n_components=2, n_neighbors=15, min_dist=0.15, metric='cosine', random_state=42)
embeddings_2d = umap_2d.fit_transform(embeddings)

topic_info = topic_model.get_topic_info()
topic_names = dict(zip(topic_info["Topic"], topic_info["CustomName"]))

def wrap(text, width=80):
    return "<br>".join(textwrap.wrap(str(text), width=width))

plot_df = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "theme": [topic_names.get(t, "Outlier") for t in topics],
    "title": df["title"].fillna("").values,
    "primary_area": df["primary_area"].fillna("").values,
    "abstract": [wrap(a) for a in df["abstract"].fillna("").values],
})

fig = px.scatter(
    plot_df, x="x", y="y",
    color="theme",
    hover_name="title",
    hover_data={
        "primary_area": True,
        "abstract": True,
        "theme": False,
        "x": False,
        "y": False,
    },
    title="CCN 2026 Abstract Topic Space",
    template="simple_white",
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=1150, height=780,
)
fig.update_traces(marker=dict(size=7, opacity=0.9))
fig.update_layout(
    legend=dict(title="Topic", x=1.01),
    hoverlabel=dict(font_size=12, namelength=-1),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)
fig.show()
# save html version
fig.write_html("ccn_2026_clusters.html")